# Leçon 17 - Créer des agents IA locaux avec Foundry Local et Qwen

Dans ce carnet, vous créez un **assistant d'ingénierie local** qui s'exécute entièrement sur votre poste de travail. Aucune inférence cloud n'est utilisée à aucun moment. L'assistant peut :

1. **Appeler des outils** via l'appel de fonction Qwen grâce à Foundry Local.
2. **Lister et lire des fichiers** dans un répertoire de projet isolé.
3. **Analyser le code** avec des métriques simples.
4. **Rechercher dans la documentation** avec un RAG local (Chroma).
5. **Utiliser un serveur MCP local** (ignoré gracieusement s'il n'est pas configuré).

Le code de l'agent est presque identique à celui des leçons dans le cloud — seul le point de terminaison client passe du cloud à `localhost`.


## Configuration

Avant d'exécuter ce notebook :

1. **Installez Microsoft Foundry Local** (voir la [documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) pour votre système d'exploitation).
2. **Téléchargez et lancez un modèle Qwen :**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Installez les paquets Python ci-dessous.

Tout s'exécute localement. Une machine avec environ 8 Go de RAM est un minimum réaliste.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Se connecter à Foundry Local

`FoundryLocalManager` télécharge le modèle si nécessaire, démarre le service local, et nous fournit un **point de terminaison compatible avec OpenAI**. Nous dirigeons ensuite le SDK OpenAI standard vers celui-ci. La clé API est un espace réservé local — aucune identification cloud n'est impliquée.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Outils locaux (Opérations de fichiers en bac à sable)

Nous créons un petit projet d'exemple sur le disque, puis définissons des outils limités à la racine de ce projet. La vérification du bac à sable est importante même localement : un outil qui lit des chemins arbitraires s'exécute avec les autorisations de votre utilisateur et peut toucher tout ce que vous pouvez.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG local avec Chroma

Nous intégrons un petit ensemble d'extraits de documentation dans une collection Chroma locale. Chroma s'exécute en processus et stocke les vecteurs sur disque — pas de serveur, pas de cloud. L'outil `search_docs` récupère les extraits les plus pertinents pour une requête.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. La boucle d’appel d’outils

Maintenant, nous enregistrons les outils auprès du modèle en utilisant le schéma d’outils OpenAI et exécutons la boucle standard d’appel d’outils — le modèle demande un outil, nous l’exécutons localement, renvoyons le résultat, et répétons jusqu’à ce que le modèle produise une réponse finale. L’appel de fonction fiable de Qwen est ce qui permet à cela de fonctionner sur l’appareil.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP Local (Optionnel)

MCP est un transport, pas un service cloud — un serveur MCP peut fonctionner comme un processus local via `stdio`. La cellule ci-dessous montre comment vous connecter à un serveur MCP local si vous en avez un configuré (par exemple un serveur de système de fichiers). Elle s'exécute gracieusement lorsque `LOCAL_MCP_COMMAND` n'est pas défini, donc le notebook fonctionne toujours de bout en bout sans lui.

Note de sécurité : un serveur MCP local fonctionne avec les permissions de votre utilisateur. Limitez-le à un répertoire de projet et validez ses sorties avant d'agir en fonction de celles-ci.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Résumé

Vous avez construit un assistant d’ingénierie qui fonctionne entièrement sur votre machine :

- **Foundry Local** servait un modèle **Qwen** derrière un point de terminaison compatible OpenAI — ainsi le code de l’agent correspond aux leçons cloud.
- **Les outils en bac à sable** ont donné à l’agent un accès aux fichiers et une analyse de code sans quitter un répertoire de projet.
- **Chroma** a fourni un **RAG local** sur la documentation.
- **Local MCP** a montré comment réutiliser l’écosystème MCP hors ligne.

Aucune inférence cloud n’a été utilisée à aucun moment.

### Défi

Étendre l’agent local pour :

1. **Travailler avec plusieurs serveurs MCP** — connecter un serveur de système de fichiers et un serveur git et laisser l’agent choisir entre eux.
2. **Utiliser la mémoire locale** — conserver un historique court des conversations sur disque afin que l’assistant se souvienne des tours précédents entre les redémarrages du carnet.
3. **Prendre en charge l’orchestration multi-agent locale** — ajouter un deuxième agent local (par ex. un réviseur) et faire collaborer les deux sur une tâche.

Dans la leçon suivante, vous apprendrez comment sécuriser les agents IA déployés.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
